In [ ]:
# Cell 0 — Setup
import os, sys, subprocess
from pathlib import Path

# ── MODE SWITCH ──────────────────────────────────────────────
SYMBOLIC_ONLY = True   # True = pure symbolic (no GPU needed), False = full neural+symbolic
# ─────────────────────────────────────────────────────────────

IS_KAGGLE = os.path.exists("/kaggle")
if IS_KAGGLE and not SYMBOLIC_ONLY:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                          "transformers", "huggingface_hub", "faiss-cpu", "sentencepiece"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if not SYMBOLIC_ONLY:
    from huggingface_hub import snapshot_download
    HF_REPO = "Dharun72/KaggleComp"
    HF_CACHE = Path("/kaggle/working/hf_cache" if IS_KAGGLE else "./hf_cache")
    HF_CACHE.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {HF_REPO} from HuggingFace...")
    import time; t0 = time.time()
    EMB_DATASET = Path(snapshot_download(repo_id=HF_REPO, repo_type="model",
                                          local_dir=str(HF_CACHE),
                                          local_dir_use_symlinks=False))
    print(f"Downloaded to {EMB_DATASET} [{time.time()-t0:.1f}s]")
else:
    EMB_DATASET = None
    print("SYMBOLIC_ONLY mode — skipping HF download")

if IS_KAGGLE:
    _comp_a = Path("/kaggle/input/competitions/llm-agentic-legal-information-retrieval")
    _comp_b = Path("/kaggle/input/llm-agentic-legal-information-retrieval")
    DATA_DIR = _comp_a if _comp_a.exists() else _comp_b
    def _find_model(hint):
        for d in Path("/kaggle/input").iterdir():
            if not d.is_dir(): continue
            if hint.lower() in d.name.lower():
                for sub in d.rglob("config.json"): return sub.parent
            for sub in d.rglob("config.json"):
                if hint.lower() in str(sub).lower(): return sub.parent
        return None
    LLM_DIR = _find_model("gemma") if not SYMBOLIC_ONLY else None
    OPUS_MT_DIR = _find_model("opus-mt") if not SYMBOLIC_ONLY else None
    OUT_PATH = Path("/kaggle/working/submission.csv")
else:
    _ROOT = Path(r"C:\Users\Dharun prasanth\OneDrive\Documents\Projects\LLm_Agentic")
    DATA_DIR = _ROOT/"Data"
    LLM_DIR = None
    OPUS_MT_DIR = _ROOT/"models"/"opus-mt-en-de" if not SYMBOLIC_ONLY else None
    OUT_PATH = _ROOT/"submission.csv"

E5_FT_DIR = EMB_DATASET/"e5-legal-finetuned" if EMB_DATASET else None
RERANK_DIR = EMB_DATASET/"reranker-legal-finetuned" if EMB_DATASET else None

RERANK_K = 60; SEED = 42
TASK_INSTRUCTION = "Given a legal question, retrieve relevant Swiss law articles and court decisions."
SMOKE = os.environ.get("SMOKE","0") == "1"

print(f"\nIS_KAGGLE={IS_KAGGLE}  SYMBOLIC_ONLY={SYMBOLIC_ONLY}  SMOKE={SMOKE}")
for n,p in [("Data",DATA_DIR),("Embeddings",EMB_DATASET),("E5-FT",E5_FT_DIR),
            ("Reranker",RERANK_DIR),("Gemma",LLM_DIR),("OpusMT",OPUS_MT_DIR)]:
    st = "[OK]" if p and Path(str(p)).exists() else "[SKIP]" if SYMBOLIC_ONLY else "[MISS]"
    print(f"  {st} {n}: {p}")

In [ ]:
# Cell 1 — Load data (+ embeddings/FAISS only when not SYMBOLIC_ONLY)
import torch, time, gc, re, warnings, json as _json
import numpy as np, pandas as pd
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

t0 = time.time()
laws = pd.read_csv(DATA_DIR/"laws_de.csv").fillna("")
val = pd.read_csv(DATA_DIR/"val.csv").fillna("")
test = pd.read_csv(DATA_DIR/"test.csv").fillna("")
train = pd.read_csv(DATA_DIR/"train.csv").fillna("")

HAS_TITLE = "title" in laws.columns

if SYMBOLIC_ONLY:
    # Memory-efficient: stream court CSV for citations only
    print("Loading court citations (streaming, citation column only)...")
    court_cits_list = []
    for chunk in pd.read_csv(DATA_DIR/"court_considerations.csv", usecols=["citation"], chunksize=500_000):
        court_cits_list.extend(chunk["citation"].fillna("").tolist())
    court = None  # don't load full text in symbolic mode
    print(f"Data loaded [{time.time()-t0:.1f}s] | Laws:{len(laws):,} Court cits:{len(court_cits_list):,} Val:{len(val)} Test:{len(test)}")
else:
    import faiss
    court = pd.read_csv(DATA_DIR/"court_considerations.csv").fillna("")
    court_cits_list = court["citation"].tolist()
    print(f"Data loaded [{time.time()-t0:.1f}s] | Laws:{len(laws):,} Court:{len(court):,} Val:{len(val)} Test:{len(test)}")

    if DEVICE.type == "cuda":
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {p.name}, {p.total_mem/1e9:.1f}GB")

    print("\nLoading pre-computed embeddings...")
    t0 = time.time()
    court_embs = np.load(str(EMB_DATASET/"court_embeddings.npy")).astype(np.float32)
    law_embs = np.load(str(EMB_DATASET/"law_embeddings.npy")).astype(np.float32)
    court_cit_arr = np.load(str(EMB_DATASET/"court_citations.npy"), allow_pickle=True)
    law_cit_arr = np.load(str(EMB_DATASET/"law_citations.npy"), allow_pickle=True)
    print(f"  Court: {court_embs.shape} | Laws: {law_embs.shape} [{time.time()-t0:.1f}s]")

    print("Building FAISS indices...")
    t0 = time.time()
    court_faiss = faiss.IndexFlatIP(court_embs.shape[1]); court_faiss.add(court_embs)
    law_faiss = faiss.IndexFlatIP(law_embs.shape[1]); law_faiss.add(law_embs)
    print(f"  Court FAISS: {court_faiss.ntotal:,} | Law FAISS: {law_faiss.ntotal:,} [{time.time()-t0:.1f}s]")
    del court_embs, law_embs; gc.collect()

    graph_path = EMB_DATASET/"citation_graph.json"
    if graph_path.exists():
        with open(str(graph_path)) as f: citation_graph = _json.load(f)
        print(f"  Citation graph: {len(citation_graph):,} statutes")
    else:
        citation_graph = {}

# Queries
target_ids = val["query_id"].tolist() + test["query_id"].tolist()
queries_en = {}
for _,r in pd.concat([val[["query_id","query"]],test[["query_id","query"]]]).iterrows():
    queries_en[r["query_id"]] = r["query"]
if SMOKE: target_ids = target_ids[:5]
print(f"Queries: {len(target_ids)}")

In [ ]:
# Cell 2 — Helpers: canonicalize, FR-DE, 4-pass regex, abbreviation index, domain KW, co-citation, dual TF-IDF
print("Building indices...")
t0 = time.time()

# ── FR→DE abbreviation mapping (37 entries, merged from competitor) ──
FR_TO_DE = {
    "LAI":"IVG","LPGA":"ATSG","LAA":"UVG","LAMal":"KVG","LAVS":"AHVG","LPP":"BVG",
    "LACI":"AVIG","LTF":"BGG","CPC":"ZPO","CPP":"StPO","CP":"StGB","CC":"ZGB",
    "CO":"OR","LP":"SchKG","LDIP":"IPRG","LDA":"URG","LCD":"UWG","LPM":"MSchG",
    "LIFD":"DBG","LIVA":"MWSTG","LBA":"BankG","LEI":"AIG","LAsi":"AsylG","PA":"VwVG",
    "LCR":"SVG","LAT":"RPG","LArm":"ArG","LPN":"NHG","LRFP":"PrHG","LSA":"VAG",
    "LFINMA":"FINMAG","OJ":"BGG","PCF":"ZPO","LDFR":"BGBB",
    "LASuivD":"AIG","LFors":"ZPO","LREC":"BGG","LTrWap":"WG",
}
LAW_CODE_ALIASES = dict(FR_TO_DE)

def canonicalize(s):
    if not isinstance(s, str): return ""
    s = s.strip().replace("\u00df","ss").replace("\u2011","-").replace("\u2010","-").replace("\u2013","-").replace("\u00a0"," ")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\b(Art|Abs|lit|Ziff)\.\s*(\d)", r"\1. \2", s)
    for fr, de in LAW_CODE_ALIASES.items():
        s = re.sub(rf"\b{re.escape(fr)}\b", de, s)
    return s.strip()

# ── Build citation lookups ──
laws_cit_to_idx = {}
for idx, row in laws.iterrows():
    c = canonicalize(str(row["citation"]))
    if c and c not in laws_cit_to_idx: laws_cit_to_idx[c] = idx

laws_art_code_index = defaultdict(list)
for cit_str, idx in laws_cit_to_idx.items():
    parts = cit_str.split()
    if len(parts) >= 3 and parts[0] == "Art.":
        laws_art_code_index[(parts[1], parts[-1])].append(idx)

court_cit_to_idx = {}
for idx, cit in enumerate(court_cits_list):
    c = canonicalize(str(cit))
    if c and c not in court_cit_to_idx: court_cit_to_idx[c] = idx

COURT_ID_OFFSET = len(laws_cit_to_idx)
corpus_set = set(laws_cit_to_idx.keys()) | set(court_cit_to_idx.keys())
q_en_canon = {qid: canonicalize(queries_en[qid]) for qid in target_ids}

# ── KGTG blacklist ──
BLACKLIST = {c for c in corpus_set if "KGTG" in c}
print(f"  Blacklist: {len(BLACKLIST)} KGTG citations removed")

# ── Abbreviation set from corpus ──
abbrev_set = set()
for c in corpus_set:
    if c.startswith("Art."):
        parts = c.split()
        if parts: abbrev_set.add(parts[-1])
print(f"  Abbreviation set: {len(abbrev_set)} law codes")

# ── 4-pass regex extraction (competitor's advanced approach) ──
_ALL_CODES = (
    r"(?:ZGB|OR|StGB|StPO|BGG|SchKG|BV|DBG|MWSTG|ZPO|AHVG|IVG|KVG|UVG|BVG|"
    r"VVG|ATSG|ArG|DSG|IPRG|AVIG|AIG|OHG|EMRK|ELG|FZG|BankG|GwG|FusG|KG|"
    r"LugUe|SVG|AsylG|MSchG|PatG|URG|RPG|USG|StBOG|BGFA|ParlG|EnG|MStG|JStG|"
    r"VwVG|UWG|NHG|PrHG|VAG|FINMAG|BGBB|HMG|BetmG|FIDLEG|RAG|VStG|FamZG|WG|"
    r"JStPO|GBV|EIMP|BEHG|KAG|FinfraG|FINIG|BEG|VZAE|EOG|DesG|MSchV|AHVV|UVV|HVUV|"
    r"LAI|LPGA|LAA|LAMal|LAVS|LPP|LACI|LTF|CPC|CPP|CP|CC|CO|LP|LDIP|LDA|LCD|LPM|"
    r"LIFD|LIVA|LBA|LEI|LAsi|LCR|PA|LASuivD|LFors|LREC|LTrWap)"
)
_ART_FULL = re.compile(
    r"(?:Art(?:ikel|icle)?\.?)\s*(\d+[a-z]?)(?:\s+(?:Abs|al|para)\.?\s*(\d+))?(?:\s+(?:lit|let)\.?\s*([a-z]))?\s+" + _ALL_CODES,
    re.IGNORECASE
)
_ART_BARE = re.compile(
    r"(?:Art(?:ikel|icle)?\.?)\s*(\d+[a-z]?)(?:\s+(?:Abs|al|para)\.?\s*(\d+))?(?:\s+(?:lit|let)\.?\s*([a-z]))?",
    re.IGNORECASE
)
_ABBR_PAT = re.compile(r"\b" + _ALL_CODES + r"\b")
_BGE_PAT = re.compile(r"BGE\s+\d+\s+[IVX]+[a-z]?\s+\d+(?:\s+E\.\s*\d+)?")
_CASE_PAT = re.compile(r"\d[A-Z]{1,3}_\d+/\d{4}(?:\s+E\.\s*\d+)?")

def _normalize_abbrev(raw):
    return FR_TO_DE.get(raw, raw)

def _try_canon(art, abs_, abbrev):
    abbrev = _normalize_abbrev(abbrev)
    if abs_:
        c1 = f"Art. {art} Abs. {abs_} {abbrev}"
        if c1 in corpus_set and c1 not in BLACKLIST: return c1
    c2 = f"Art. {art} {abbrev}"
    if c2 in corpus_set and c2 not in BLACKLIST: return c2
    return None

def extract_regex_advanced(text):
    hits = []
    # Pass 1: Full article refs with explicit abbreviation
    for m in _ART_FULL.finditer(text):
        art, abs_, lit = m.group(1), m.group(2), m.group(3)
        abbrev = m.group(0).split()[-1]
        c = _try_canon(art, abs_, abbrev)
        if c: hits.append(c)

    # Pass 2: Bare article refs — pair with nearest abbreviation within 150 chars
    abbr_positions = [(m.start(), _normalize_abbrev(m.group(0))) for m in _ABBR_PAT.finditer(text)]
    for m in _ART_BARE.finditer(text):
        already = any(m.start() == fm.start() for fm in _ART_FULL.finditer(text))
        if already: continue
        art, abs_ = m.group(1), m.group(2)
        best_abbr, best_dist = None, 151
        for pos, abbr in abbr_positions:
            dist = abs(pos - m.start())
            if dist < best_dist and abbr in abbrev_set:
                best_dist = dist; best_abbr = abbr
        if best_abbr:
            c = _try_canon(art, abs_, best_abbr)
            if c: hits.append(c)

    # Pass 3: BGE refs (with optional E. subsection)
    for m in _BGE_PAT.finditer(text):
        c = canonicalize(m.group(0))
        if c in corpus_set and c not in BLACKLIST: hits.append(c)

    # Pass 4: Case number refs
    for m in _CASE_PAT.finditer(text):
        c = canonicalize(m.group(0))
        if c in corpus_set and c not in BLACKLIST: hits.append(c)

    return list(dict.fromkeys(hits))

def extract_abbrevs(text):
    abbrevs = []
    for m in _ABBR_PAT.finditer(text):
        a = _normalize_abbrev(m.group(0))
        if a in abbrev_set and a not in abbrevs:
            abbrevs.append(a)
    return abbrevs

# ── Abbreviation index: abbrev → sorted citations by global frequency ──
global_cit_freq = Counter()
cit_cooccur = defaultdict(Counter)
for _df in [train, val]:
    for _, _r in _df.iterrows():
        cits = [c.strip() for c in str(_r.get("gold_citations", "")).split(";") if c.strip()]
        s = set(cits)
        for c in s: global_cit_freq[c] += 1
        for c in cits:
            for o in s:
                if o != c: cit_cooccur[c][o] += 1

global_score = {c: v for c, v in global_cit_freq.items() if c not in BLACKLIST and c in corpus_set}

abbrev_to_cits = defaultdict(list)
for c in corpus_set:
    if c.startswith("Art.") and c not in BLACKLIST:
        parts = c.split()
        if parts: abbrev_to_cits[parts[-1]].append(c)
for k in abbrev_to_cits:
    abbrev_to_cits[k].sort(key=lambda c: global_score.get(c, 0), reverse=True)

def abbrev_expand(direct_hits, abbrevs, top_per=20):
    expanded = []
    for abbr in abbrevs:
        candidates = abbrev_to_cits.get(abbr, [])
        scored = []
        for cit in candidates:
            co = sum(cit_cooccur.get(d, {}).get(cit, 0) for d in direct_hits)
            g = global_score.get(cit, 0)
            scored.append((cit, co * 5 + g))
        scored.sort(key=lambda x: x[1], reverse=True)
        for cit, _ in scored[:top_per]:
            if cit not in expanded: expanded.append(cit)
    return expanded

# ── Co-citation expansion ──
def coexpand(seeds, top=25):
    scores = Counter()
    seed_set = set(seeds)
    for c in seeds:
        w = 1 + global_score.get(c, 0) * 0.05
        for o, cnt in cit_cooccur.get(c, {}).items():
            if o not in BLACKLIST and o in corpus_set:
                scores[o] += cnt * w
    for c in seed_set: scores.pop(c, None)
    return list(seed_set) + [c for c, _ in scores.most_common(top) if c in corpus_set]

# ── Domain keywords (~130 entries, merged from competitor) ──
DOMAIN_KW = {
    "testament":["ZGB"],"testator":["ZGB"],"holograph":["ZGB"],"will ":["ZGB"],
    "bequest":["ZGB"],"legatee":["ZGB"],"intestate":["ZGB"],"disinherit":["ZGB"],
    "forced share":["ZGB"],"reserved":["ZGB"],
    "heir":["ZGB","IPRG"],"inherit":["ZGB","IPRG"],"estate":["ZGB"],
    "custody":["ZGB","ZPO"],"divorce":["ZGB","ZPO"],"maintenance":["ZGB"],
    "child support":["ZGB"],"parental":["ZGB"],"alimony":["ZGB"],"spouse":["ZGB"],
    "visitation":["ZGB"],"matrimoni":["ZGB"],"marital":["ZGB"],"separation":["ZGB"],
    "overnight":["ZGB"],
    "invalidity":["IVG","ATSG"],"invalide":["IVG","ATSG"],"disability":["IVG","ATSG"],
    "rehab":["IVG"],"earning capac":["IVG","ATSG"],"work capac":["IVG","ATSG"],
    "insured":["IVG","UVG","KVG"],"occupational":["BVG","UVG"],
    "asthma":["UVG","IVG"],"allerg":["UVG","IVG"],"respiratory":["UVG","IVG"],
    "granulomatos":["UVG","IVG"],
    "pension":["BVG","AHVG"],"accident":["UVG"],"social insur":["ATSG"],
    "ahv":["AHVG"],"contribution":["AHVG","BVG"],
    "criminal":["StGB","StPO"],"detention":["StPO"],"pre-trial":["StPO"],"pretrial":["StPO"],
    "theft":["StGB"],"assault":["StGB"],"robbery":["StGB"],"fraud":["StGB"],
    "sentence":["StGB"],"conviction":["StGB"],"sexual":["StGB"],"juvenile":["JStG"],
    "collusion":["StPO"],"flight risk":["StPO"],"accused":["StPO","StGB"],
    "offence":["StGB"],"offense":["StGB"],"disloyal":["StGB"],"embezzl":["StGB"],
    "dna profile":["StPO","StGB"],
    "contract":["OR"],"lease":["OR"],"tenancy":["OR"],"landlord":["OR"],"tenant":["OR"],
    "employer":["OR","ArG"],"employee":["OR","ArG"],"employment":["OR","ArG"],
    "damages":["OR"],"liability":["OR"],"mandate":["OR"],
    "work contract":["OR"],"defect":["OR"],"product liab":["OR","PrHG"],
    "bank":["BankG","OR"],"forgery":["StGB"],"forged":["StGB"],"signature":["OR"],
    "board member":["OR"],
    "bankrupt":["SchKG","OR"],"insolvenc":["SchKG"],"creditor":["SchKG"],
    "debt enforc":["SchKG"],"opposition":["SchKG"],"lien":["SchKG","ZGB"],
    "mortgage":["ZGB","SchKG"],
    "appeal":["BGG"],"federal court":["BGG"],"jurisdiction":["BGG","ZPO"],
    "supreme court":["BGG"],"provisional":["ZPO"],"interim":["ZPO"],
    "injunction":["ZPO"],"evidence":["ZPO","StPO"],
    "international":["IPRG"],"foreign judgm":["IPRG"],"choice of law":["IPRG"],
    "habitual resi":["IPRG"],"recognition":["IPRG"],"applicable law":["IPRG"],
    "trademark":["MSchG"],"copyright":["URG"],"unfair comp":["UWG"],
    "domain name":["MSchG","UWG"],
    "tax":["DBG","MWSTG"],"withholding":["DBG","VStG"],
    "asylum":["AsylG","AIG"],"residence permit":["AIG"],
    "traffic":["SVG"],"driving":["SVG"],
    "land register":["ZGB"],"real estate":["ZGB"],"building":["RPG"],
    "construct":["OR","RPG"],"contractor":["OR"],"statutory lien":["ZGB"],
    "medical":["KVG","UVG"],"physician":["KVG"],"treatment":["KVG","UVG"],
    "ophthalm":["KVG"],"surgeon":["KVG"],"standard of care":["KVG","OR"],
}

def get_domain_abbrevs(text):
    text_lower = text.lower()
    codes = set()
    for kw, code_list in DOMAIN_KW.items():
        if kw in text_lower: codes.update(code_list)
    return list(codes)

# ── Macro-F1 ──
def macro_f1(gold_sets, pred_sets):
    f1s = []
    for g, p in zip(gold_sets, pred_sets):
        g = set(g) if not isinstance(g, set) else g
        p = set(p) if not isinstance(p, set) else p
        if not g and not p: f1s.append(1.0); continue
        if not g or not p: f1s.append(0.0); continue
        tp = len(g & p)
        if tp == 0: f1s.append(0.0); continue
        pr, rc = tp/len(p), tp/len(g)
        f1s.append(2*pr*rc/(pr+rc))
    return float(np.mean(f1s)) if f1s else 0.0

# ── Dual TF-IDF: word + char n-gram (fitted on ALL queries for better IDF) ──
train_queries = train["query"].tolist()
train_golds = [str(r.get("gold_citations","")).split(";") for _, r in train.iterrows()]
all_queries = train_queries + [queries_en[q] for q in target_ids]

tfidf_w = TfidfVectorizer(max_features=80_000, ngram_range=(1,2), sublinear_tf=True,
                           strip_accents="unicode", min_df=1)
tfidf_c = TfidfVectorizer(max_features=60_000, ngram_range=(3,5), analyzer="char_wb",
                           sublinear_tf=True, min_df=2)

tfidf_w.fit(all_queries)
tfidf_c.fit(all_queries)
train_wv = tfidf_w.transform(train_queries)
train_cv = tfidf_c.transform(train_queries)

print(f"Indices built [{time.time()-t0:.1f}s] | Laws:{len(laws_cit_to_idx):,} Court:{len(court_cit_to_idx):,}")
print(f"  CoOccur:{len(cit_cooccur):,} | GlobalScore:{len(global_score):,} | AbbrIdx:{len(abbrev_to_cits):,}")
print(f"  TF-IDF word:{train_wv.shape} char:{train_cv.shape}")

In [ ]:
# Cell 3 — Translation (skip in SYMBOLIC_ONLY mode)
if SYMBOLIC_ONLY:
    translations = {qid: "" for qid in target_ids}
    print("SYMBOLIC_ONLY — skipping translation")
else:
    print("\n=== Translation ===")
    t0 = time.time()
    USE_LLM = LLM_DIR is not None and Path(str(LLM_DIR)).exists()
    USE_OPUS = not USE_LLM and OPUS_MT_DIR is not None and Path(str(OPUS_MT_DIR)).exists()

    if USE_LLM:
        from transformers import AutoTokenizer as _ATok, AutoModelForCausalLM as _ACLM
        print(f"Loading Gemma from {LLM_DIR}...")
        _tok = _ATok.from_pretrained(str(LLM_DIR))
        _mdl = _ACLM.from_pretrained(str(LLM_DIR), device_map="auto", dtype=torch.float16)
        _mdl.eval()
        print(f"  Loaded [{time.time()-t0:.1f}s]")
        def _gen(prompt, max_tok=300):
            try: t = _tok.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
            except: t = prompt
            inp = _tok(t, return_tensors="pt").to(_mdl.device)
            with torch.no_grad():
                out = _mdl.generate(**inp, max_new_tokens=max_tok, temperature=0.3, do_sample=True,
                                    top_p=0.9, pad_token_id=_tok.eos_token_id)
            return _tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        translations = {}
        for qid in target_ids:
            prompt = ("Translate the following English legal question into German. "
                      "Output ONLY the German translation, nothing else.\n\n"
                      f"English: {queries_en[qid][:600]}\n\nGerman:")
            translations[qid] = canonicalize(_gen(prompt, max_tok=300))
        del _mdl, _tok; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
        print(f"Gemma done [{time.time()-t0:.1f}s]")
    elif USE_OPUS:
        from transformers import AutoTokenizer as _ATok, MarianMTModel
        _opus_tok = _ATok.from_pretrained(str(OPUS_MT_DIR))
        _opus_mdl = MarianMTModel.from_pretrained(str(OPUS_MT_DIR)).to(DEVICE).eval()
        @torch.no_grad()
        def _tr(texts, bs=8):
            r=[]
            for i in range(0,len(texts),bs):
                b=texts[i:i+bs]; e=_opus_tok(b,return_tensors="pt",padding=True,truncation=True,max_length=512).to(DEVICE)
                g=_opus_mdl.generate(**e,max_new_tokens=512,num_beams=4)
                r.extend([_opus_tok.decode(x,skip_special_tokens=True) for x in g])
            return r
        translations = {qid:canonicalize(de) for qid,de in zip(target_ids,_tr([queries_en[q] for q in target_ids]))}
        del _opus_mdl,_opus_tok; gc.collect()
        if DEVICE.type=="cuda": torch.cuda.empty_cache()
    else:
        translations = {qid: canonicalize(queries_en[qid]) for qid in target_ids}
        print("No translation model, using English")

In [ ]:
# Cell 4 — Symbolic scoring: 6-stage weighted pipeline (competitor-style)
print("\n=== Symbolic Scoring ===")
t0 = time.time()

def resolve_cit_str(cit_str):
    if cit_str in laws_cit_to_idx: return ("law", cit_str)
    if cit_str in court_cit_to_idx: return ("court", cit_str)
    return None

def score_query(query, qid):
    scored = {}

    # Stage 1: Direct regex hits — weight 10.0
    direct = extract_regex_advanced(query)
    for c in direct:
        if c not in BLACKLIST: scored[c] = scored.get(c, 0) + 10.0

    # Stage 2: Abbreviation expansion — weight 3.0
    abbrevs = extract_abbrevs(query)
    exp = abbrev_expand(direct, abbrevs, top_per=25)
    for c in exp:
        if c not in scored: scored[c] = scored.get(c, 0) + 3.0

    # Stage 3: Domain keyword expansion — weight 1.5
    domain_ab = get_domain_abbrevs(query)
    extra_ab = list(set(domain_ab) - set(abbrevs))
    dom_exp = abbrev_expand(direct + list(scored.keys())[:5], extra_ab, top_per=15)
    for c in dom_exp:
        if c not in scored: scored[c] = scored.get(c, 0) + 1.5

    # Stage 4: Training transfer (dual TF-IDF) — weight 1.5
    qvec_w = tfidf_w.transform([query])
    qvec_c = tfidf_c.transform([query])
    sw = cosine_similarity(qvec_w, train_wv).flatten()
    sc = cosine_similarity(qvec_c, train_cv).flatten()
    combined_sim = 0.5 * sw + 0.5 * sc
    top_idx = np.argsort(combined_sim)[::-1][:50]
    transfer_scores = Counter()
    for i in top_idx:
        sim = float(combined_sim[i])
        if sim < 0.02: continue
        for cit in train_golds[i]:
            cit = cit.strip()
            if cit and cit not in BLACKLIST and cit in corpus_set:
                transfer_scores[cit] += sim * sim
    for c, s in transfer_scores.most_common(80):
        scored[c] = scored.get(c, 0) + 1.5 * min(s, 1.0)

    # Stage 5: Global frequency micro-boost — weight 0.03
    for c in list(scored.keys()):
        scored[c] += global_score.get(c, 0) * 0.03

    # Stage 6: Co-citation on top-15 seeds — weight via global score
    top15 = sorted(scored, key=scored.get, reverse=True)[:15]
    exp2 = coexpand(top15, top=30)
    for c in exp2:
        if c not in scored:
            scored[c] = global_score.get(c, 0) * 0.005

    return scored

# Score all queries
all_scored = {}
entity_hits = {}   # keep for diagnostics
cocite_hits = {}
transfer_hits = {}
domain_hits_diag = {}

for qid in target_ids:
    query = queries_en[qid]
    scored = score_query(query, qid)
    all_scored[qid] = scored

    # Also track per-signal hits for val diagnostics
    direct = extract_regex_advanced(query)
    entity_hits[qid] = [c for c in direct if c in corpus_set]

    abbrevs = extract_abbrevs(query)
    exp = abbrev_expand(direct, abbrevs, top_per=25)
    cocite_exp = coexpand(direct + exp[:5], top=30)
    cocite_hits[qid] = cocite_exp

    domain_ab = get_domain_abbrevs(query)
    domain_hits_diag[qid] = domain_ab

    qvec_w = tfidf_w.transform([query])
    qvec_c = tfidf_c.transform([query])
    sw = cosine_similarity(qvec_w, train_wv).flatten()
    sc = cosine_similarity(qvec_c, train_cv).flatten()
    combined_sim = 0.5 * sw + 0.5 * sc
    top_idx = np.argsort(combined_sim)[::-1][:50]
    t_cits = []
    for i in top_idx:
        if combined_sim[i] < 0.02: continue
        for cit in train_golds[i]:
            cit = cit.strip()
            if cit and cit in corpus_set: t_cits.append(cit)
    transfer_hits[qid] = list(dict.fromkeys(t_cits))[:50]

    if len(all_scored) <= 3 or len(all_scored) % 10 == 0:
        print(f"  {qid}: regex={len(direct)} abbrev={len(abbrevs)} domain={len(domain_ab)} transfer={len(transfer_hits[qid])} total={len(scored)}", flush=True)

print(f"Symbolic scoring done [{time.time()-t0:.1f}s]")
print(f"  Total scored citations: {sum(len(s) for s in all_scored.values()):,}")

In [ ]:
# Cell 5 — Dense retrieval (skip in SYMBOLIC_ONLY mode)
if SYMBOLIC_ONLY:
    print("SYMBOLIC_ONLY — skipping dense retrieval")
    dense_law_results = {qid: [] for qid in target_ids}
    dense_court_results = {qid: [] for qid in target_ids}
else:
    import torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModel
    HAS_E5 = E5_FT_DIR is not None and Path(str(E5_FT_DIR)).exists()
    if HAS_E5:
        print(f"\n=== Fine-tuned E5 Query Encoding ===")
        t0 = time.time()
        e5_tok = AutoTokenizer.from_pretrained(str(E5_FT_DIR))
        e5_mdl = AutoModel.from_pretrained(str(E5_FT_DIR), torch_dtype=torch.float16).to(DEVICE).eval()
        print(f"Loaded [{time.time()-t0:.1f}s]")
        def mean_pool(output, mask):
            emb = output.last_hidden_state
            mask_exp = mask.unsqueeze(-1).expand(emb.size()).float()
            return (emb * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
        @torch.no_grad()
        def encode_queries(texts, bs=32, ml=512):
            embs = []
            for i in range(0, len(texts), bs):
                b = texts[i:i+bs]
                enc = e5_tok(b, return_tensors="pt", padding=True, truncation=True, max_length=ml).to(DEVICE)
                with torch.amp.autocast("cuda"):
                    out = e5_mdl(**enc)
                    emb = mean_pool(out, enc["attention_mask"])
                    emb = F.normalize(emb, p=2, dim=1)
                embs.append(emb.float().cpu().numpy())
            return np.vstack(embs).astype(np.float32)
        q_texts_en = [f"query: Instruct: {TASK_INSTRUCTION} Query: {q_en_canon[q]}" for q in target_ids]
        q_texts_de = [f"query: Instruct: {TASK_INSTRUCTION} Query: {translations.get(q, q_en_canon[q])}" for q in target_ids]
        print("Encoding queries (EN + DE)...")
        en_e = encode_queries(q_texts_en); de_e = encode_queries(q_texts_de)
        qe = (en_e + de_e) / 2.0
        qe = qe / np.maximum(np.linalg.norm(qe, axis=1, keepdims=True), 1e-9)
        LAWS_K = 100; COURT_K = 100
        D_law, I_law = law_faiss.search(qe, LAWS_K)
        D_court, I_court = court_faiss.search(qe, COURT_K)
        dense_law_results = {qid: [int(i) for i in I_law[qi] if i>=0] for qi,qid in enumerate(target_ids)}
        dense_court_results = {qid: [int(i)+COURT_ID_OFFSET for i in I_court[qi] if i>=0] for qi,qid in enumerate(target_ids)}
        del e5_mdl, e5_tok, en_e, de_e, qe; gc.collect()
        if DEVICE.type=="cuda": torch.cuda.empty_cache()
        print(f"Dense done [{time.time()-t0:.1f}s]")
    else:
        dense_law_results = {qid: [] for qid in target_ids}
        dense_court_results = {qid: [] for qid in target_ids}

In [ ]:
# Cell 6 — Citation graph (skip in SYMBOLIC_ONLY mode)
if SYMBOLIC_ONLY:
    print("SYMBOLIC_ONLY — skipping citation graph")
    graph_hits = {qid: [] for qid in target_ids}
else:
    print("\n=== Citation Graph Expansion ===")
    t0 = time.time()
    graph_hits = {}
    for qid in target_ids:
        seed_law_cits = set()
        for fid in list(entity_hits.get(qid, [])) + list(cocite_hits.get(qid, [])):
            if isinstance(fid, str):
                if fid in laws_cit_to_idx: seed_law_cits.add(fid)
            elif fid < COURT_ID_OFFSET:
                seed_law_cits.add(canonicalize(str(law_cit_arr[fid])))
        graph_courts = set()
        for cit in seed_law_cits:
            court_indices = citation_graph.get(cit, [])
            for ci in court_indices[:10]:
                graph_courts.add(ci + COURT_ID_OFFSET)
        graph_hits[qid] = list(graph_courts)
    print(f"Graph done [{time.time()-t0:.1f}s]")

In [ ]:
# Cell 7 — Fusion: weighted scores (symbolic) or tier-ordered (neural)
print("\n=== Fusion ===")
t0 = time.time()

# Ranked results: sorted by weighted score descending
ranked_results = {}
for qid in target_ids:
    if SYMBOLIC_ONLY:
        scored = all_scored[qid]
        ranked = sorted(scored.keys(), key=lambda c: scored[c], reverse=True)
        ranked_results[qid] = ranked
    else:
        # Neural mode: keep tier ordering, same as before
        seen = set(); combined = []
        for fid in entity_hits.get(qid, []):
            c = resolve_citation(fid) if callable(globals().get('resolve_citation')) else str(fid)
            if c not in seen: seen.add(c); combined.append(c)
        for c in cocite_hits.get(qid, []):
            if c not in seen: seen.add(c); combined.append(c)
        for c in transfer_hits.get(qid, []):
            if c not in seen: seen.add(c); combined.append(c)
        ranked_results[qid] = combined

print(f"Fusion done [{time.time()-t0:.1f}s]")
for qid in target_ids[:3]:
    top5 = ranked_results[qid][:5]
    sc5 = [f"{all_scored[qid].get(c,0):.2f}" for c in top5] if SYMBOLIC_ONLY else ["—"]*5
    print(f"  {qid}: top5={top5} scores={sc5} total={len(ranked_results[qid])}")

In [ ]:
# Cell 8 — Reranker (skip in SYMBOLIC_ONLY mode)
if SYMBOLIC_ONLY:
    print("SYMBOLIC_ONLY — skipping reranker, using weighted score ordering")
    reranked_cits = ranked_results
else:
    HAS_RR = RERANK_DIR is not None and Path(str(RERANK_DIR)).exists()
    if HAS_RR:
        print(f"\n=== Reranker: {RERANK_DIR} ===")
        t0 = time.time()
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        rtok = AutoTokenizer.from_pretrained(str(RERANK_DIR))
        rmdl = AutoModelForSequenceClassification.from_pretrained(str(RERANK_DIR)).to(DEVICE).eval()

        def resolve_text(fid, ml=1500):
            if fid >= COURT_ID_OFFSET:
                ci = fid - COURT_ID_OFFSET
                if ci < len(court): r = court.iloc[ci]; return f"{r['citation']} {str(r['text'])[:ml]}"
                return ""
            if fid < len(laws):
                r = laws.iloc[fid]
                title = str(r.get('title','')) if HAS_TITLE else ''
                return f"{r['citation']} {title} {str(r['text'])[:ml]}".strip()
            return ""

        @torch.no_grad()
        def rerank(qen, ctexts, bs=32):
            sc = []
            for i in range(0, len(ctexts), bs):
                b = ctexts[i:i+bs]
                e = rtok([qen]*len(b), b, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
                sc.extend(rmdl(**e).logits.squeeze(-1).cpu().float().numpy().tolist())
            return np.array(sc)

        reranked_cits = {}
        for qid in target_ids:
            rc = ranked_results[qid][:RERANK_K]
            if not rc: reranked_cits[qid] = []; continue
            ctxts = [resolve_text(laws_cit_to_idx.get(c, court_cit_to_idx.get(c, -1))) for c in rc]
            sc = rerank(q_en_canon[qid], ctxts)
            reranked_cits[qid] = [rc[i] for i in np.argsort(-sc)]
        del rmdl, rtok; gc.collect()
        if DEVICE.type=="cuda": torch.cuda.empty_cache()
        print(f"Reranked [{time.time()-t0:.1f}s]")
    else:
        reranked_cits = ranked_results

In [ ]:
# Cell 9 — Val F1 + adaptive K calibration
print("\n=== Val Evaluation ===")

def calibrate(vr, k_candidates=[5,8,10,12,15,20,25,30,35,40,45,50,60,70,80,100]):
    best_f1, best_k = 0.0, 10
    gs = [r["gold"] for r in vr]
    for k in k_candidates:
        ps = [set(r["rc"][:k]) for r in vr]
        f1 = macro_f1(gs, ps)
        if f1 > best_f1: best_f1, best_k = f1, k
    return best_k, best_f1

vr = []
for _, row in val.iterrows():
    qid = row["query_id"]
    if qid not in reranked_cits: continue
    gold = set(canonicalize(c) for c in str(row["gold_citations"]).split(";") if c.strip())
    rc = reranked_cits[qid]
    n_regex = len(entity_hits.get(qid, []))
    n_abbr = len(extract_abbrevs(queries_en[qid])) if qid in queries_en else 0
    vr.append({"qid": qid, "gold": gold, "rc": rc, "n_regex": n_regex, "n_abbr": n_abbr})

if vr:
    best_k, best_f1 = calibrate(vr)
    print(f"\n*** Val macro-F1 = {best_f1:.4f} @ top-{best_k} ***\n")

    for r in vr:
        p = set(r["rc"][:best_k]); g = r["gold"]; tp = len(g & p)
        pr = tp/len(p) if p else 0; rc_ = tp/len(g) if g else 0
        f1 = 2*pr*rc_/(pr+rc_) if (pr+rc_) > 0 else 0
        print(f"  {r['qid']}: F1={f1:.3f} TP={tp} pred={best_k} gold={len(g)}")
        # Show what we got right and what we missed
        if tp > 0: print(f"    HIT: {list(g & p)[:5]}")
        missed = g - p
        if missed: print(f"    MISS: {list(missed)[:5]}")

    # Per-signal oracle F1
    print("\nPer-signal F1 (oracle coverage at best_k):")
    for nm, fn in [
        ("regex_direct", lambda q: entity_hits.get(q, [])),
        ("cocite", lambda q: cocite_hits.get(q, [])),
        ("transfer", lambda q: transfer_hits.get(q, [])),
    ]:
        gs, ps = [], []
        for r in vr:
            gs.append(r["gold"])
            cits = fn(r["qid"])
            ps.append(set(cits[:best_k]) if isinstance(cits, list) else set())
        print(f"  {nm}: F1={macro_f1(gs, ps):.4f}")

    # Coverage: what % of gold citations appear anywhere in our ranked list
    print("\nCoverage (gold citations in full ranked list):")
    for r in vr:
        full_set = set(r["rc"])
        covered = len(r["gold"] & full_set)
        print(f"  {r['qid']}: {covered}/{len(r['gold'])} = {covered/len(r['gold'])*100:.0f}%")
else:
    best_k = 10
    print("No val queries found")

In [ ]:
# Cell 10 — Submission with adaptive K
print("\n=== Submission ===")

def adaptive_k(qid, base=best_k):
    query = queries_en.get(qid, "")
    d = len(extract_regex_advanced(query))
    a = len(extract_abbrevs(query))
    if d >= 5 or a >= 3: return min(base + 20, 100)
    if d == 0 and a == 0: return max(base, 20)
    return base

# Global frequency fallback: most common training citations
fallback_cits = [c for c, _ in sorted(global_score.items(), key=lambda x: x[1], reverse=True)[:20]]

rows = []
tids = set(test["query_id"].tolist())
for qid in target_ids:
    if qid not in tids: continue
    k = adaptive_k(qid)
    rc = reranked_cits.get(qid, [])
    seen = set(); dd = []
    for c in rc[:k]:
        if c and c not in seen and c not in BLACKLIST:
            seen.add(c); dd.append(c)
    if not dd: dd = fallback_cits[:k]
    rows.append({"query_id": qid, "predicted_citations": ";".join(dd)})

sub = pd.DataFrame(rows)
missing = set(test["query_id"]) - set(sub["query_id"])
print(f"Rows: {len(sub)}" + (f" MISSING:{missing}" if missing else " [OK]"))
sub.to_csv(OUT_PATH, index=False)
print(f"Written to {OUT_PATH}")

# Print prediction stats
pred_counts = [len(r["predicted_citations"].split(";")) for _, r in sub.iterrows()]
print(f"  Avg predictions/query: {np.mean(pred_counts):.1f} (min={min(pred_counts)}, max={max(pred_counts)})")

In [ ]:
# Cell 11 — Summary
print("\n" + "="*60)
if SYMBOLIC_ONLY:
    print("v60: SYMBOLIC-ONLY (no GPU, no neural models)")
    print("="*60)
    print(f"  Laws: {len(laws):,} | Court cits: {len(court_cits_list):,}")
    print(f"  Signals: 4-pass regex -> abbrev expand -> domain KW -> dual TF-IDF transfer -> global freq -> co-citation")
    print(f"  Regex: {sum(1 for h in entity_hits.values() if h)}/{len(target_ids)} queries with direct hits")
    print(f"  Domain KW: ~{len(DOMAIN_KW)} keywords")
    print(f"  FR->DE: {len(FR_TO_DE)} mappings")
    print(f"  Blacklist: {len(BLACKLIST)} KGTG citations")
    print(f"  TF-IDF: word(80k) + char(60k) dual transfer")
else:
    print("v60: FULL NEURAL + SYMBOLIC PIPELINE")
    print("="*60)
    print(f"  Laws: {len(laws):,} | Court: {len(court):,}")
    print(f"  Dense + Symbolic + Reranker")
if vr:
    print(f"  Val F1: {best_f1:.4f} @ top-{best_k}")
print("="*60)